# Model Context Protocol

Uma ferramenta escrita como função Python só existe no programa que a define. O Model Context Protocol (MCP) coloca ferramentas, recursos e prompts em um processo à parte, o servidor, que qualquer cliente compatível descobre e usa. Este notebook escreve um servidor de viagem, conecta-o a um agente, acrescenta servidores da comunidade e termina com um agente geral que combina servidores e skills.

In [ ]:
# No Google Colab, descomente e rode uma vez.

# !pip install -q "langchain[mcp]" langchain-openai langgraph uv

# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
import json
import logging
import os
import subprocess
import sys
from pathlib import Path

os.environ["FASTMCP_LOG_LEVEL"] = "WARNING"
logging.getLogger("mcp.client.stdio").setLevel(logging.CRITICAL)  # servidores antigos escrevem linhas em branco

import pandas as pd
import yaml
from fastmcp import Client
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.mcp import MCPAdapter
from langchain.messages import HumanMessage
from langchain.tools import tool

pd.set_option("display.max_colwidth", None)

In [ ]:
model = init_chat_model("openai:gpt-5.4-mini", temperature=0.0)

## Servidor MCP

O `FastMCP` cria o servidor e registra cada primitiva com um decorador, que tira nome, descrição e parâmetros da função. As três primitivas diferem em quem decide usá-las:

- **ferramentas** (`@mcp.tool`), que o modelo chama;
- **recursos** (`@mcp.resource`), que a aplicação lê por uma URI e anexa ao contexto;
- **prompts** (`@mcp.prompt`), modelos de mensagem que o usuário escolhe.

In [ ]:
%%writefile travel_server.py
import json
import urllib.request
from urllib.parse import quote

from fastmcp import FastMCP

mcp = FastMCP("viagem")


def fetch(url: str) -> dict:
    """Faz um GET e devolve o JSON da resposta."""
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=30) as response:
        return json.load(response)


@mcp.tool()
def get_weather(city: str) -> str:
    """Devolve a previsão de temperatura e chuva dos próximos três dias em uma cidade."""
    place = fetch(f"https://geocoding-api.open-meteo.com/v1/search?name={quote(city)}&count=1&language=pt")["results"][0]
    daily = fetch(
        f"https://api.open-meteo.com/v1/forecast?latitude={place['latitude']}&longitude={place['longitude']}"
        "&daily=temperature_2m_min,temperature_2m_max,precipitation_probability_max&forecast_days=3&timezone=auto"
    )["daily"]
    days = zip(*daily.values())
    return f"{place['name']}, {place['country']}\n" + "\n".join(f"{day}: {low} a {high} °C, chuva {rain}%" for day, low, high, rain in days)


@mcp.tool()
def convert_currency(amount: float, source: str, target: str) -> dict[str, float]:
    """Converte um valor entre moedas pelo código de três letras, como BRL, CNY ou USD."""
    rates = fetch(f"https://api.frankfurter.dev/v1/latest?amount={amount}&from={source}&to={target}")["rates"]
    return {source: amount, target: rates[target]}


@mcp.resource("viagem://checklist")
def packing_checklist() -> str:
    """Itens para levar em uma viagem internacional."""
    return "passaporte\nseguro viagem\nadaptador de tomada\ncópia das reservas"


@mcp.resource("viagem://guia/{city}")
def city_guide(city: str) -> str:
    """Resumo da Wikipédia em português sobre uma cidade."""
    return fetch(f"https://pt.wikipedia.org/api/rest_v1/page/summary/{quote(city)}")["extract"]


@mcp.prompt()
def plan_trip(city: str, days: str, budget: str) -> str:
    """Pede um roteiro de viagem que leve em conta o clima e o orçamento."""
    return (
        f"Monte um roteiro de {days} dias em {city} com orçamento de {budget}. "
        "Consulte a previsão do tempo, converta o orçamento para a moeda local "
        "e sugira uma atividade por dia, em no máximo seis frases."
    )


# sem transporte, o servidor conversa pela entrada e pela saída padrão
mcp.run(show_banner=False)

## Comunicação entre cliente e servidor

O MCP define como cliente e servidor trocam mensagens, independentemente da linguagem em que cada um foi escrito. Esta parte conversa com o servidor de viagem primeiro à mão, mensagem por mensagem, e depois pelo cliente da biblioteca.

### Transporte local

O servidor é um programa comum: rodado com `python travel_server.py`, ele lê mensagens na entrada padrão e responde na saída padrão, no transporte chamado **stdio**. Cada mensagem é uma linha de **JSON-RPC**: um pedido leva `id`, método e parâmetros, a resposta volta com o mesmo `id`, e uma notificação não tem `id` nem resposta. As funções abaixo iniciam o servidor como subprocesso e fazem o papel do cliente, escrevendo e lendo essas linhas à mão.

In [ ]:
process = subprocess.Popen([sys.executable, "travel_server.py"], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)


def write(message: dict):
    """Escreve uma mensagem JSON-RPC na entrada padrão do servidor."""
    process.stdin.write(json.dumps({"jsonrpc": "2.0"} | message) + "\n")
    process.stdin.flush()


def read() -> dict:
    """Lê a próxima resposta na saída padrão do servidor."""
    return json.loads(process.stdout.readline())

### Descoberta

O protocolo não abre sessão: cada pedido leva no campo `_meta` um envelope com a versão do protocolo e as capacidades do cliente, e o servidor responde a cada um de forma independente. O pedido `server/discover` devolve o nome do servidor, as versões que ele aceita e as suas **capacidades**, isto é, as primitivas que oferece. Assim o cliente sabe de antemão o que pode pedir: um servidor sem `prompts` nas capacidades não recebe `prompts/list`.

In [ ]:
meta = {"io.modelcontextprotocol/protocolVersion": "2026-07-28", "io.modelcontextprotocol/clientCapabilities": {}}

write({"id": 1, "method": "server/discover", "params": {"_meta": meta}})
print(read())

write({"id": 2, "method": "tools/call", "params": {"name": "convert_currency", "arguments": {"amount": 2000, "source": "BRL", "target": "CNY"}, "_meta": meta}})
print(read())
process.terminate()

O servidor se apresentou como `viagem`, informou a versão que aceita e declarou `tools`, `resources` e `prompts`. O `tools/call` devolveu o resultado duas vezes: como texto em `content`, que vai para o modelo, e como objeto em `structuredContent`, que o programa lê sem interpretar uma frase.

### Cliente

O `Client` do FastMCP faz esse trabalho: inicia o subprocesso, faz a descoberta, acrescenta o envelope a cada pedido e devolve objetos Python. A conexão vive dentro do `async with`, e o servidor termina quando o bloco fecha.

In [ ]:
server = Path("travel_server.py")

async with Client(server) as client:
    print(client.transport)
    print(client.server_info.name, client.server_capabilities)
    listed = await client.list_tools()

pd.DataFrame([{"name": t.name, "input_schema": t.input_schema["properties"], "output_schema": t.output_schema} for t in listed])

O `tools/list` traz nome, descrição e parâmetros em JSON Schema, o que o modelo precisa para chamar cada ferramenta. A função `request` abre um cliente do servidor de viagem por chamada e é usada nas próximas seções.

In [ ]:
async def request(method: str, *args):
    """Inicia o servidor, faz a descoberta e chama um método do cliente."""
    async with Client(server) as client:
        return await getattr(client, method)(*args)

## Primitivas no cliente

Cada primitiva tem um método para listar e outro para usar: `tools/list` e `tools/call`, `resources/list` e `resources/read`, `prompts/list` e `prompts/get`. No `Client`, cada método do protocolo vira um método assíncrono, como `list_tools` e `call_tool`.

### Ferramentas

O `call_tool` envia um `tools/call` com o nome da ferramenta e os argumentos, e o servidor executa a função.

In [ ]:
called = await request("call_tool", "get_weather", {"city": "Pequim"})

print(called.content[0].text)

A previsão foi calculada dentro do servidor, e o cliente recebeu só o texto.

### Recursos

Recursos são lidos pela URI, sempre por iniciativa da aplicação. Uma URI fixa, como `viagem://checklist`, aponta para um conteúdo único, e um template como `viagem://guia/{city}` recebe o parâmetro na própria URI.

In [ ]:
resources = await request("list_resources")
templates = await request("list_resource_templates")
guide = await request("read_resource", "viagem://guia/Pequim")

print([str(r.uri) for r in resources], [t.uri_template for t in templates])
print(guide[0].text)

O resumo de Pequim veio da Wikipédia no momento da leitura.

### Prompts

Um prompt recebe argumentos e devolve mensagens prontas para entrar na conversa.

In [ ]:
prompt = await request("get_prompt", "plan_trip", {"city": "Pequim", "days": "3", "budget": "2000 reais"})

for message in prompt.messages:
    print(message.role, ":", message.content.text)

O servidor devolveu uma mensagem de `user` pronta, que uma interface como o Claude Desktop oferece como comando.

### Exercício 1

Acrescente à célula do servidor uma ferramenta `get_holidays` que recebe o ano e o código de duas letras de um país, como CN, e devolve os feriados nacionais pela API `https://date.nager.at/api/v3/PublicHolidays/{ano}/{pais}`. Grave o arquivo de novo e chame `list_tools` e `call_tool` com a ferramenta nova. Quantos feriados a ferramenta devolve para a China em 2026? Mantenha a ferramenta no servidor, porque ela volta nos próximos exercícios.

In [ ]:
# Seu código aqui

## Servidor no agente

Até aqui o programa decidiu cada chamada. No agente, o modelo decide quando usar as ferramentas, enquanto prompts e recursos continuam nas mãos do usuário e da aplicação.

### Ferramentas do servidor no agente

O `MCPAdapter`, do módulo `langchain.mcp` (em beta), recebe o mesmo alvo do `Client` e converte cada ferramenta MCP em uma ferramenta do LangChain.

In [ ]:
async with MCPAdapter(server) as adapter:
    tools = await adapter.list_tools()

[t.name for t in tools]

Recursos não viram ferramentas: o guia da cidade entra na mensagem de sistema, por decisão da aplicação, e só as ferramentas ficam a critério do modelo. A chamada é `ainvoke` porque as ferramentas do adaptador são assíncronas.

In [ ]:
guide = await request("read_resource", "viagem://guia/Pequim")

travel_agent = create_agent(model, tools=tools, system_prompt=f"Você é um assistente de viagem.\n\nGuia da cidade:\n{guide[0].text}")
result = await travel_agent.ainvoke({"messages": [HumanMessage(
    "Monte um roteiro de 3 dias em Pequim com orçamento de 2000 reais, considerando a previsão do tempo "
    "e o orçamento na moeda local, em no máximo seis frases."
)]})

In [ ]:
for message in result["messages"]:
    print(message.type, ":", message.tool_calls if getattr(message, "tool_calls", None) else message.text)

O agente chamou as duas ferramentas do servidor e montou o roteiro com a previsão e o valor em yuans. O guia entrou no contexto sem nenhuma chamada, porque a leitura do recurso foi decisão da aplicação.

### Resultado de uma ferramenta remota

O servidor devolve só a resposta do `tools/call`. O adaptador a transforma em uma `ToolMessage`, com o texto em `content`, o `structuredContent` em `artifact` e o `tool_call_id` do pedido do modelo.

In [ ]:
tool_message = next(m for m in result["messages"] if m.type == "tool" and m.name == "convert_currency")

print(tool_message.tool_call_id)
print(tool_message.content)
print(tool_message.artifact)

Para o agente, a ferramenta remota produz a mesma `ToolMessage` que uma função local.

## Servidores da comunidade

O protocolo rende mais quando o servidor foi escrito por outra pessoa. Um agente usa vários servidores prontos ao mesmo tempo, sem nenhuma ferramenta escrita no notebook.

### Servidores remotos

Um servidor remoto roda em outra máquina e é acessado por uma URL, no transporte **Streamable HTTP**. O `Client` escolhe o transporte pelo alvo: o caminho de um script abre um subprocesso por stdio, e uma URL abre uma conexão HTTP. O servidor do Kiwi.com, que busca voos, é remoto e público.

In [ ]:
async with Client("https://mcp.kiwi.com") as remote_client:
    print(remote_client.transport)
    print([t.name for t in await remote_client.list_tools()])

Só o alvo mudou em relação ao servidor de viagem.

### Vários servidores em um agente

Os servidores locais rodam como subprocesso, com `uvx` se escritos em Python e `npx` se escritos em TypeScript: o `time` informa a hora em qualquer fuso e o `filesystem` grava arquivos. Um único dicionário reúne o Kiwi, o Exa, que busca na web, o servidor de viagem e os dois locais.

In [ ]:
servers = {
    # remotos
    "kiwi": {"url": "https://mcp.kiwi.com"},
    "exa": {"url": "https://mcp.exa.ai/mcp"},
    # locais
    "viagem": {"command": sys.executable, "args": ["travel_server.py"]},
    "time": {"command": "uvx", "args": ["mcp-server-time"]},
    "filesystem": {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-filesystem", "."]},
}

In [ ]:
hub_adapter = MCPAdapter({"mcpServers": servers})

async with hub_adapter:
    hub_tools = await hub_adapter.list_tools()

[t.name for t in hub_tools]

São 22 ferramentas de cinco servidores, prefixadas com o nome do servidor, como `time_get_current_time`. A função `show` abre o adaptador uma vez por pedido, para não relançar os servidores a cada chamada, e imprime cada passo assim que ele acontece.

In [ ]:
async def show(task: str, adapter: MCPAdapter, agent):
    """Executa o pedido e imprime as chamadas de ferramenta, o começo de cada resultado e a resposta."""
    async with adapter:
        async for update in agent.astream({"messages": [HumanMessage(task)]}, stream_mode="updates"):
            for node in update.values():
                for message in (node or {}).get("messages", []):
                    for call in getattr(message, "tool_calls", []):
                        print("chamada:", call["name"], json.dumps(call["args"], ensure_ascii=False)[:150])
                    if message.type == "tool":
                        print("  resultado:", " ".join(message.text.split())[:200])
                    elif not message.tool_calls:
                        print("\nresposta:", message.text)

### Composição

O modelo escolhe cada ferramenta pela descrição e monta os argumentos a partir do pedido. O agente recebe as ferramentas dos cinco servidores.

In [ ]:
system_prompt = "Responda em português, em no máximo três frases. Use as ferramentas sempre que o pedido envolver arquivos ou dados externos."

hub_agent = create_agent(model, tools=hub_tools, system_prompt=system_prompt)

A pergunta abaixo depende de três servidores de autores diferentes.

In [ ]:
await show(
    "Quanto custa o voo mais barato de São Paulo para Pequim em 10/11/2026, como estará o tempo em Pequim "
    "e que horas são lá agora?",
    hub_adapter,
    hub_agent,
)

O voo veio do Kiwi, a previsão do servidor de viagem e a hora do `time`, sem que nenhum servidor conhecesse os outros. Descrições e saídas de servidores de terceiros entram no prompt sem revisão, o que abre espaço para instruções maliciosas.

### Exercício 2

Acrescente a `servers` o servidor da Wikipédia, `{"command": "uvx", "args": ["wikipedia-mcp", "--language", "pt"]}`, crie um adaptador, as ferramentas e um agente novos, já com o servidor de viagem do Exercício 1, e passe-os a `show` com a pergunta "Quais são os feriados nacionais da China em 2026 e o que se comemora no Festival do Meio do Outono?". De qual servidor veio cada ferramenta chamada?

In [ ]:
# Seu código aqui

## Agent Skills

Uma **skill** ensina o agente a fazer um tipo de tarefa: é uma pasta com um `SKILL.md`, cujo frontmatter traz `name` e `description` e cujo corpo traz as instruções, no formato aberto adotado por Claude, Codex e Cursor. O conteúdo entra no contexto por **revelação progressiva**:

- o nome e a descrição ficam sempre na mensagem de sistema;
- o corpo entra quando o modelo decide carregar a skill;
- arquivos de apoio da pasta só são lidos se o corpo os citar.

O prompt do servidor é escolhido pelo usuário; a skill é escolhida pelo modelo, como uma ferramenta, mas traz instruções em vez de executar uma ação.

A primeira skill é simples e formata a previsão do tempo.

In [ ]:
!mkdir -p skills/weather-forecast skills/arxiv-report

In [ ]:
%%writefile skills/weather-forecast/SKILL.md
---
name: weather-forecast
description: Apresenta a previsão do tempo de uma cidade em tabela. Use quando o usuário perguntar sobre o tempo ou o clima em uma cidade.
---

Consulte a previsão com get_weather e responda só com uma tabela Dia | Mínima | Máxima | Chuva, seguida de uma frase de recomendação.

A segunda pesquisa artigos do arXiv com o Exa e grava um relatório com o `filesystem`.

In [ ]:
%%writefile skills/arxiv-report/SKILL.md
---
name: arxiv-report
description: Pesquisa artigos do arXiv sobre um tema e grava um relatório em Markdown. Use quando o usuário pedir um relatório, levantamento ou estado da arte sobre um tema científico.
---

1. Busque com web_search_exa usando a consulta "site:arxiv.org <tema>" e numResults 5.
2. Escolha até cinco artigos dos resultados, usando os resumos que a busca devolve.
3. Grave com write_file o arquivo relatorio-<tema>.md, com um título, um parágrafo de panorama e uma tabela Artigo | Ano | Contribuição, citando cada artigo pelo link do arXiv.
4. Responda com o nome do arquivo e um resumo de duas frases.

A ferramenta `load_skill` devolve o corpo da skill. A mensagem de sistema do agente geral traz só o catálogo, com o nome e a descrição de cada skill.

In [ ]:
@tool
def load_skill(name: str) -> str:
    """Carrega as instruções completas de uma skill pelo nome."""
    return Path("skills", name, "SKILL.md").read_text()


metadata = [yaml.safe_load(path.read_text().split("---")[1]) for path in sorted(Path("skills").glob("*/SKILL.md"))]
catalog = "\n".join(f"- {meta['name']}: {meta['description']}" for meta in metadata)
print(catalog)

In [ ]:
general_system_prompt = f"""Você é um assistente geral. Use as ferramentas sempre que o pedido depender de dados externos.

Skills disponíveis:
{catalog}

Quando o pedido corresponder a uma skill, chame load_skill antes de qualquer outra ferramenta e siga as instruções."""

general_agent = create_agent(model, tools=hub_tools + [load_skill], system_prompt=general_system_prompt)

O agente geral usa os mesmos servidores, então `show` recebe o mesmo `hub_adapter`. Cada pedido abaixo exercita um caso: nenhuma skill, a skill simples e a skill com vários servidores.

In [ ]:
await show("Quanto são 100 dólares em reais?", hub_adapter, general_agent)

In [ ]:
await show("Como estará o tempo em Pequim nos próximos dias?", hub_adapter, general_agent)

In [ ]:
await show("Faça um relatório sobre agentes que usam Model Context Protocol.", hub_adapter, general_agent)

In [ ]:
print(next(Path().glob("relatorio-*.md")).read_text())

A conversão respondeu sem carregar skill. A pergunta sobre o tempo carregou `weather-forecast` e saiu na tabela pedida, e o relatório carregou `arxiv-report`, buscou os artigos e gravou o arquivo. Servidores MCP também distribuem skills, como recursos `skill://`, e o `apps/mcp_agent` do repositório combina conectores e skills em uma interface.

### Exercício 3

Crie a pasta `skills/destination-holidays` e grave nela, com `%%writefile`, um `SKILL.md` que consulte os feriados com `get_holidays`, do Exercício 1, busque na Wikipédia o que se comemora em cada um, com o servidor do Exercício 2, e responda com uma tabela Data | Feriado | O que se comemora. Recrie o `catalog`, o `general_system_prompt` e o agente geral com as ferramentas do Exercício 2 e passe a `show` a pergunta "Vou para a China em 2026. Que feriados vou encontrar por lá?". Qual skill o agente carregou e quais ferramentas chamou?

In [ ]:
# Seu código aqui